# Characteristic Portfolio Formation

Two investors can use the same characteristic and still hold different investments. They may compare firms with the whole market or with similar-sized firms, update holdings at different dates, or assign different weights. These choices determine the portfolio return.

In this notebook, I explain how breakpoints, sorting procedures, formation dates and weights turn the measurements from [01](01_firm_characteristics.ipynb) into holdings. I then compare June assignments and construct historical characteristic-portfolio returns.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name=='notebooks' else ROOT
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from factor_asset_pricing.characteristics import fama_french_characteristics
from factor_asset_pricing.sorts import quantile_breakpoints, assign_portfolios, assign_annual_portfolios, portfolio_returns
NAVY, BLUE, TEAL, AMBER, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':12,
    'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,'legend.fontsize':10,
    'figure.dpi':110,'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'lines.linewidth':1.8,'grid.alpha':.15})

from factor_asset_pricing.wrds_data import load_prepared_data
RAW_WRDS = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"


## 1. Characteristic Portfolio Formation

### 1.1 Why form characteristic portfolios?

Characteristic portfolios define an investment comparison. A size sort compares firms with different market capitalizations, while a valuation sort compares firms with different valuation measurements. We can then examine their returns and exposures; the regressions in 05 evaluate abnormal returns.

Grouping firms also reduces some company-specific noise. The portfolios should remain distinct enough to answer the question and diversified enough to avoid relying on a few individual outcomes.

### 1.2 Investment universes and breakpoints

The **investment universe** contains the securities we can hold. The **reference universe** contains those used to calculate breakpoints. For example, NYSE cutoffs can be applied to stocks on several exchanges.

Quintiles use the reference distribution's 20th, 40th, 60th and 80th percentiles. These boundaries need not produce equally populated portfolios in a broader investment universe. Ties also affect counts, so the assignment rule must specify which group receives a value on a boundary. Missing characteristics or reference cutoffs leave securities unassigned.

### 1.3 Univariate, independent and conditional sorts

A **univariate sort** groups firms on one characteristic. The groups can still differ in other ways: a valuation sort may also compare different size compositions.

An **independent sort** calculates each characteristic's breakpoints separately, then intersects the assignments. A size × valuation grid therefore uses common valuation boundaries across size groups. Independent cutoff calculations do not imply independent characteristics or returns.

A **conditional sort** first groups firms on one characteristic, then calculates the next characteristic's boundaries within each group. A high-valuation small firm is high relative to its size group, rather than the whole reference sample. The sorting order therefore matters.

Conditional sorting can balance membership, but does not guarantee matching exposures. Additional splits can also leave fewer firms in each portfolio.

### 1.4 Formation and rebalancing

Formation assigns securities using information available at that date. A June assignment can determine membership from July through the following June; the current June return still belongs to the preceding assignment.

Rebalancing determines the weights maintained within those groups. A stock can keep its annual portfolio label while its value weight changes each month. Annual formation therefore does not mean constant weights.

Monthly formation updates labels more frequently and may increase turnover. Either design can produce monthly returns. We need to state when labels change, how weights update and how securities enter or leave.

### 1.5 Portfolio weights and returns

For portfolio $p$ and holding month $t$,

$$R_{p,t}=\sum_{i\in\mathcal P_{p,t}}w_{i,t-1}R_{i,t},\qquad\sum_iw_{i,t-1}=1.$$

$\mathcal P_{p,t}$ contains the holdings and $w_{i,t-1}$ their pre-return capital shares. A 20% position earning 5% contributes one percentage point to the portfolio return.

Equal weights give every included security the same share, while value weights use beginning-period market equity:

$$w^{EW}_{i,t-1}=\frac1{n_p},\qquad
w^{VW}_{i,t-1}=\frac{ME_{i,t-1}}{\sum_{j\in p}ME_{j,t-1}}.$$

Equal weighting gives smaller securities more relative representation. Different weights can therefore produce different returns from the same holdings. If missing holding-period returns are dropped and weights rescaled, the reported return uses information available afterward; this is a data convention rather than a complete trading rule.

For a fully invested portfolio, subtract the matched risk-free return once:

$$\text{Portfolio excess return}=R_{p,t}-R_{f,t}.$$

Dates, frequency and units must agree. A long–short difference cancels the common risk-free component, as explained in 03.

## 2. Example: Fama–French Characteristic Portfolios

I first form June-2020 size × BM portfolios and compare independent and conditional assignments. I then construct historical size × BM, profitability and investment portfolios to examine their returns.

### 2.1 Data and formation conventions

The example uses the linked panel and measurement rules from 01. The June comparison requires positive market equity and usable BM for NYSE, AMEX and Nasdaq securities. Eligible NYSE stocks define the quintile boundaries applied to the full eligible universe.

June size, fiscal-2019 book equity and December 2019 market equity determine the assignments. The accounts come directly from the annual inputs prepared in 00. July availability remains an assumption because historical publication dates are unavailable. The local universe and accounting conventions can differ from official French eligibility.

In [ ]:
FORMATION_YEAR = 2020
formation_month = pd.Timestamp(FORMATION_YEAR, 6, 1)
formation_date = pd.Timestamp(FORMATION_YEAR, 6, 30)
december_month = pd.Timestamp(FORMATION_YEAR-1, 12, 1)
raw, original_annual = load_prepared_data(PROCESSED, RAW_WRDS,
    start=f"{FORMATION_YEAR-1}-01-01", end=formation_month)
assert not raw.duplicated(["permno", "mdate"]).any()
# Independent annual inputs, not accounts recovered from a future July row.
annual = original_annual.loc[
    original_annual.fyear.isin([FORMATION_YEAR-2, FORMATION_YEAR-1]) &
    original_annual.available_from.le(pd.Timestamp(FORMATION_YEAR, 7, 1))
].copy()
assert not annual.duplicated(["gvkey", "fyear"]).any()
api_panel = raw.rename(columns={"date": "crsp_date", "mdate": "date"})
measured = fama_french_characteristics(api_panel, annual_data=annual)
formation = measured.loc[measured.date.eq(formation_month) & measured.exchcd.isin([1,2,3]) &
                          measured.me.gt(0), ['permno','gvkey','date','exchcd','me','me_dec','bm','op','inv']].copy()
formation['size_millions'] = formation.me/1000
formation = formation.sort_values('permno').reset_index(drop=True)
assert len(formation)>0 and formation.permno.is_unique
measure_columns = ['size_millions','bm','op','inv']
print(f'June {FORMATION_YEAR}: {len(formation):,} eligible security records in the locally prepared linked universe.')
print('Independent annual records use the assumed July holding-availability convention, not verified filing dates.')


### 2.2 From characteristics to portfolio assignments

The independent $5\times5$ sort uses four NYSE boundaries for size and four for BM. The library uses the lower observed value when a percentile falls between observations, and values exactly on a boundary enter the lower group.

The table shows these boundaries. I then compare the independent assignments with size-conditional BM assignments for the same securities. The conditional design recalculates BM boundaries within each size group, changing the reference comparison while holding the universe fixed.

In [ ]:
eligible = formation.loc[formation.bm.notna()].copy()
chars = ['size_millions','bm']
labels = ['size_millions_portfolio','bm_portfolio']
reference = eligible.loc[eligible.exchcd.eq(1)]
assert len(reference)>0
probabilities = np.arange(1,5)/5
cuts = {name:quantile_breakpoints(reference[name],5) for name in chars}
breakpoint_table = pd.DataFrame({'Percentile (%)':100*probabilities,
    'June size ($m)':cuts['size_millions'],'Book-to-market (ratio)':cuts['bm']}).set_index('Percentile (%)')
print(f'{len(eligible):,} eligible June securities, including {len(reference):,} NYSE reference securities.')
display(breakpoint_table.round(6))
independent = assign_portfolios(eligible,chars,5,reference='nyse',exchange_col='exchcd')
conditional = assign_portfolios(eligible,chars,5,reference='nyse',exchange_col='exchcd',
                               conditional_on={'bm':['size_millions']})
# Independently select order statistics and assign boundaries with searchsorted.
for name in chars:
    values = np.sort(reference[name].to_numpy(dtype=float))
    hand_cuts = values[np.floor((len(values)-1)*probabilities).astype(int)]
    np.testing.assert_allclose(cuts[name],hand_cuts)
    expected = np.searchsorted(hand_cuts,eligible[name].to_numpy(dtype=float),side='left')+1
    np.testing.assert_array_equal(independent[name+'_portfolio'].to_numpy(dtype=int),expected)
for size_group in range(1,6):
    group = conditional.loc[conditional.size_millions_portfolio.eq(size_group)]
    local_reference = group.loc[group.exchcd.eq(1),'bm'].to_numpy(dtype=float)
    assert len(local_reference)>0
    values = np.sort(local_reference)
    boundaries = values[np.floor((len(values)-1)*probabilities).astype(int)]
    expected = np.searchsorted(boundaries,group.bm.to_numpy(dtype=float),side='left')+1
    np.testing.assert_array_equal(group.bm_portfolio.to_numpy(dtype=int),expected)
counts = {name:assigned.groupby(labels,observed=True).size().unstack(fill_value=0)
          .reindex(index=range(1,6),columns=range(1,6),fill_value=0)
          for name,assigned in [('Independent',independent),('Conditional',conditional)]}
assert all(int(table.to_numpy().sum())==len(eligible) for table in counts.values())
changed = int((independent.bm_portfolio != conditional.bm_portfolio).sum())
print(f'{changed:,} securities change BM group when the comparison becomes conditional on size.')
fig, axes = plt.subplots(1,2,figsize=(10,4.3),layout='constrained')
maximum = max(table.to_numpy().max() for table in counts.values())
for ax,(name,table) in zip(axes,counts.items()):
    im=ax.imshow(table.to_numpy(),cmap='Blues',vmin=0,vmax=maximum,aspect='auto')
    for (row,col),value in np.ndenumerate(table.to_numpy()):
        ax.text(col,row,str(value),ha='center',va='center',color='white' if value>maximum*.55 else NAVY)
    ax.set(xticks=range(5),xticklabels=range(1,6),yticks=range(5),yticklabels=range(1,6),
        xlabel='BM group (low → high)',ylabel='Size group (small → big)',title=name)
fig.colorbar(im,ax=axes,label='Number of eligible security records',shrink=.8)
fig.suptitle('June 2020: common versus size-conditional BM boundaries',fontsize=13)
plt.show()


Both panels contain the same 2,925 eligible securities, but 1,631 change BM group when moving from independent to conditional sorting. The size assignments stay fixed. The difference comes from using common BM boundaries in one design and size-specific boundaries in the other.

NYSE reference quintiles need not produce equal counts across the broader holdings universe. More balanced counts also do not guarantee neutral factor exposures.

### 2.3 Historical characteristic-portfolio returns

I repeat the June assignments to construct historical $5\times5$ portfolios for size × BM, profitability and investment. NYSE quintiles use the same lower-observed cutoffs, with membership held from July through June. Value weights require market equity from exactly the previous calendar month, and June formation requires the preceding December observation.

The figure shows mean monthly total returns. Size increases down the rows and the other characteristic across columns. All grids share a color scale, while each portfolio uses its available returns. Coverage is reported separately for each family.

In [ ]:
from factor_asset_pricing.construction import characteristic_panel, annual_test_portfolios
from factor_asset_pricing.french_data import load_french
from factor_asset_pricing.validation import validate_portfolios
# Build the full history using the independent annual source table.
panel_history, annual_history = load_prepared_data(PROCESSED, RAW_WRDS)
measured_history = characteristic_panel(panel_history, annual_history)
portfolio_frames = []
for family, characteristic in [("size_bm", "bm"), ("size_op", "op"), ("size_inv", "inv")]:
    returns, breaks = annual_test_portfolios(measured_history, characteristic)
    returns["family"] = family
    portfolio_frames.append(returns)
portfolios = pd.concat(portfolio_frames, ignore_index=True)
assert not portfolios.duplicated(["family", "mdate", "row_portfolio", "column_portfolio"]).any()
assert portfolios.ret.dropna().ge(-1).all()


In [ ]:
for family in ['size_bm','size_op','size_inv']:
    sample = portfolios.loc[portfolios.family.eq(family)]
    observed = sample.loc[sample.ret.notna()]
    counts_by_cell = observed.groupby(['row_portfolio','column_portfolio']).size()
    print(f'{family}: {observed.mdate.min():%Y-%m}–{observed.mdate.max():%Y-%m}, '
          f'{observed.mdate.nunique()} observed months, '
          f'{counts_by_cell.min()}–{counts_by_cell.max()} observations per cell.')


In [ ]:
means={}
for family,label in [("size_bm","Book-to-market"),("size_op","Profitability"),("size_inv","Investment")]:
    data=portfolios.loc[portfolios.family.eq(family)]
    means[label]=data.pivot_table(index="row_portfolio",columns="column_portfolio",values="ret",aggfunc="mean").astype(float)*100
fig,axes=plt.subplots(1,3,figsize=(12.5,4),constrained_layout=True)
lower=min(t.min().min() for t in means.values())
upper=max(t.max().max() for t in means.values())
for ax,(label,table) in zip(axes,means.items()):
    image=ax.imshow(table,cmap="RdBu_r",aspect="auto",vmin=lower,vmax=upper)
    for (r,c),value in np.ndenumerate(table.to_numpy()): ax.text(c,r,f"{value:.2f}",ha="center",va="center",fontsize=7)
    ax.set(title=f"Size × {label}",xlabel="Characteristic portfolio (low → high)",ylabel="Size portfolio (small → big)",xticks=range(5),xticklabels=range(1,6),yticks=range(5),yticklabels=range(1,6))
fig.colorbar(image,ax=axes,shrink=.75,label="Average monthly return (%)")
fig.suptitle("Average returns across characteristic portfolios",weight="bold")
plt.show()


Among the smallest stocks, mean returns rise from roughly 0.73% to 1.35% per month between the lowest and highest BM groups. The corresponding endpoints among the largest stocks are much closer, at about 0.99% and 1.01%.

High-profitability portfolios earn higher average returns than low-profitability portfolios in each size row. Low-investment portfolios also outperform the highest-investment portfolios on average, although the interior groups are not strictly ordered. These are total-return comparisons, not alphas or evidence of a single economic explanation. Differences in family coverage also matter when comparing the grids.

[03](03_factor_construction.ipynb) combines selected portfolio contrasts into long–short factors. [05](05_time_series_factor_regressions.ipynb) then evaluates their benchmark exposures and abnormal returns.

### 2.4 Comparison with published portfolios

I compare each constructed portfolio with its published value-weighted French counterpart on matching dates. Correlation measures co-movement, mean differences compare average returns, and Newey–West regression diagnostics assess the relationship after adjusting for a fitted exposure.

The linked universe, accounting rules, source vintages and lagged-capitalization weights can produce differences from the published portfolios. The comparison checks agreement without assuming exact replication.

In [ ]:
validation_rows = []
for family in ["size_bm", "size_op", "size_inv"]:
    published = load_french(family, ROOT / "data/raw/french").rename_axis("mdate").reset_index().melt(id_vars="mdate", var_name="asset", value_name="ret")
    published[["row_portfolio", "column_portfolio"]] = published.asset.str.extract(r"S(\d)C(\d)").astype(int)
    comparison = validate_portfolios(portfolios.loc[portfolios.family.eq(family)], published,
        portfolio_cols=["row_portfolio", "column_portfolio"], date_col="mdate", hac_lags=12)
    comparison["family"] = family
    validation_rows.append(comparison)
portfolio_validation = pd.concat(validation_rows, ignore_index=True)
display(portfolio_validation.groupby("family").agg(
    portfolios=("correlation", "size"), median_correlation=("correlation", "median"),
    min_correlation=("correlation", "min"), mean_return_difference=("mean_difference", "mean")))


## References

- Fama, E. F., and French, K. R. (2015). A five-factor asset pricing model. *Journal of Financial Economics*, 116(1), 1–22. [DOI](https://doi.org/10.1016/j.jfineco.2014.10.010).
- Kenneth French, [five-factor construction documentation](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).
